In [ ]:
import numpy as np
import random

In [ ]:
class GridWorld:
    def __init__(self, size=4, start=(0,0), goal=(3,3)):
       self.size = size
       self.start = start
       self.goal = goal
       self.state = start
       self.action = [(0,1),(1,0),(0,-1),(-1,0)]
    def reset(self):
        self.state = self.start
        return self.state

    def step(self,action):
        next_state = (self.state[0]+ self.action[action][0],
                      self.state[1]+ self.action[action][1])
        if 0 <= next_state[0] < self.size and 0 <= next_state[1] < self.size:
            self.state = next_state

        reward = 1 if self.state == self.goal else -0.1
        done = self.state == self.goal
        return self.state, reward, done



In [ ]:
def monte_carlo_control(env, episodes=5000, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}
    returns = {((i, j), a): [] for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        episode = []

        while True:
            action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break

        G = 0
        for state, action, reward in reversed(episode):
            G = gamma * G + reward
            if((state, action) not in [(x[0], x[1]) for x in episode[:-1]]):
                returns[(state, action)].append(G)
                Q[(state, action)] = np.mean(returns[(state, action)])

    return Q





In [ ]:
def sarsa(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])

        while True:
            next_state, reward, done = env.step(action)
            next_action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(next_state, a)])

            # Update Q-value
            Q[(state, action)] += alpha * (reward + gamma * Q[(next_state, next_action)] - Q[(state, action)])

            state, action = next_state, next_action
            if done:
                break

    return Q

In [ ]:
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()

        while True:
            action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[(state, a)])
            next_state, reward, done = env.step(action)

            Q[(state, action)] += alpha * (reward + gamma * max(Q[(next_state, a)] for a in range(4)) - Q[(state, action)])
            state = next_state

            if done:
                break

    return Q

In [ ]:
env = GridWorld()
Q_mc = monte_carlo_control(env)
Q_sarsa = sarsa(env)
Q_qlearning = q_learning(env)

In [ ]:
print("Sample Q-values from Monte Carlo:", {k: Q_mc[k] for k in list(Q_mc.keys())[:20]})
print("Sample Q-values from SARSA:", {k: Q_sarsa[k] for k in list(Q_sarsa.keys())[:20]})
print("Sample Q-values from Q-Learning:", {k: Q_qlearning[k] for k in list(Q_qlearning.keys())[:20]})

Sample Q-values from Monte Carlo: {((0, 0), 0): 0, ((0, 0), 1): 0, ((0, 0), 2): 0, ((0, 0), 3): 0, ((0, 1), 0): 0, ((0, 1), 1): 0, ((0, 1), 2): 0, ((0, 1), 3): 0, ((0, 2), 0): 0, ((0, 2), 1): 0, ((0, 2), 2): 0, ((0, 2), 3): 0, ((0, 3), 0): 0, ((0, 3), 1): 0, ((0, 3), 2): 0, ((0, 3), 3): 0, ((1, 0), 0): 0, ((1, 0), 1): 0, ((1, 0), 2): 0, ((1, 0), 3): 0}
Sample Q-values from SARSA: {((0, 0), 0): 0.09139589576292696, ((0, 0), 1): 0.11395267373148169, ((0, 0), 2): 0.006334239314255521, ((0, 0), 3): 0.0031237907287047398, ((0, 1), 0): 0.19057047278847528, ((0, 1), 1): 0.25341972346822117, ((0, 1), 2): -0.010238127859698656, ((0, 1), 3): 0.13401027904860577, ((0, 2), 0): 0.25312258822952544, ((0, 2), 1): 0.40557604116987006, ((0, 2), 2): 0.04518965387826761, ((0, 2), 3): 0.12463965990761415, ((0, 3), 0): -0.03635442253900001, ((0, 3), 1): 0.5597545767705754, ((0, 3), 2): -0.03376198198000001, ((0, 3), 3): -0.03916990000000001, ((1, 0), 0): 0.2594786175912437, ((1, 0), 1): 0.19333664917583454